In [ ]:
import pandas as pd

file_path = "../data/raw/Online Retail.xlsx"
df = pd.read_excel(file_path)

df.head()

#df.shape # 행 개수와 열 개수 확인
#df.columns # 열 이름 목록 확인
#df.info() # 전체 구조와 데이터타입, 결측치 여부 요약
#df.describe(include='all') # 숫자형 열의 기초 통계랑을 보여주나, include='all' 사용 시 문자열 열까지 포험
#df.isna().sum() # 각 열별 결측치(NULL) 개수
#df.duplicated().sum() # 중복된 행의 개수

(541909, 8)

In [ ]:
# csv 파일 변환

input_path = "../data/raw/Online Retail.xlsx"
output_path = "../data/processed/online_retail.csv"

# xlsx에서 csv로 변환된 복사본 생성
df.to_csv(output_path, index = False, encoding = 'utf-8-sig')

csv_df = pd.read_csv("../data/processed/online_retail.csv")

print(csv_df.shape)
print(csv_df.head())
print(csv_df.dtypes)

(541909, 8)
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

           InvoiceDate  UnitPrice  CustomerID         Country  
0  2010-12-01 08:26:00       2.55     17850.0  United Kingdom  
1  2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
2  2010-12-01 08:26:00       2.75     17850.0  United Kingdom  
3  2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
4  2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
InvoiceNo       object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
UnitPrice      float64
CustomerID     float64
Country   

원본 파일이 xlsx 형식이므로, csv 형식의 파일로 변환하여 저장하였다.
csv가 분석 도구들과 더 잘 연결되고, 구조가 단순해서 다루기 편하기 때문이다.

In [ ]:
# 결측치 확인

csv_df.isna().sum()

# 결측치의 비율
(csv_df.isna().mean() * 100).round(2)

InvoiceNo       0.00
StockCode       0.00
Description     0.27
Quantity        0.00
InvoiceDate     0.00
UnitPrice       0.00
CustomerID     24.93
Country         0.00
dtype: float64

Description은 전체 행의 0.27%가 NULL값이다 - 결측률은 낮지만 해당 행의 거래 특성을 추가 확인한다.
CustomerID는 전체 행의 24.93%가 NULL값이다. - 해당 행은 전체 매출분석에는 포함할 수 있지만, 고객 식별이 필요한 재구매, RFM(Recency, Frequency, Monetary) 분석에서는 제외해야 한다.

In [ ]:
# 중복값 확인

# 각 행을 조사하여 중복 행을 표시, 주문 번호, 상품 코드, 주문 일시 순으로 정렬
# keep=False의 경우 여러 개의 중복 행 중 첫 행 또한 중복 행으로 표시

csv_df.duplicated().sum()
csv_df[csv_df.duplicated(keep=False)].sort_values(
    by = ['InvoiceNo', 'StockCode', 'InvoiceDate']
).head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom


In [ ]:
# 날짜 확인

# dtype('O') - Object 형식이라는 것을 확인, 날짜가 아니라 문자열로 읽히는 상태
csv_df['InvoiceDate'].dtype

dtype('O')

In [ ]:
# 날짜형으로 변환

# 날짜형으로 변환하는 과정이며, errors = 'coerce' 사용 시 변환되지 않는 값은 NaT로 입력된다.
csv_df['InvoiceDate'] = pd.to_datetime(
    csv_df['InvoiceDate'],
    errors = 'coerce'
)

csv_df['InvoiceDate'].dtype

dtype('<M8[ns]')

In [ ]:
# 변환된 값의 결측치 확인
csv_df['InvoiceDate'].isna().sum()

0

In [16]:
csv_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


## Table Relationships

customers
- CustomerID
- Country

orders
- InvoiceNo
- InvoiceDate
- CustomerID

products
- StockCode
- Description

order_items
- InvoiceNo
- StockCode
- Quantity
- UnitPrice

Relationships:
- customers.CustomerID -> orders.CustomerID
- orders.InvoiceNo -> order_items.InvoiceNo
- products.StockCode -> order_items.StockCode